# Producto aplicado 2 – Pipeline de IA para predicción y segmentación

**Asignatura:** SM26_CIA6041 – AI and Organizational Data Management  
**Estudiante:** Felipe Gerardo Rodríguez Cordero  
**Dataset:** Online Retail – UCI Machine Learning Repository  
**Entrega final:** Semana 10

## Objetivo
Desarrollar un pipeline reproducible de análisis de datos que incluya ETL, análisis exploratorio (EDA), un modelo supervisado comparado con una línea base y un modelo no supervisado de segmentación de clientes.

El análisis mantiene la continuidad del caso Online Retail trabajado en las semanas anteriores.

## Instrucciones de ejecucion
Abra el notebook en Google Colab y seleccione **Entorno de ejecucion > Ejecutar todas**. Cargue el Excel original Online Retail cuando se solicite.
Esta copia corrige el orden de carga y conserva las salidas de la ejecucion original. La ejecucion completa desde cero de esta version debe comprobarse en Colab.


In [ ]:
import pandas as pd
from pathlib import Path

# En Colab: cargar el Excel original si no esta en la sesion.
archivo = Path("Online Retail.xlsx")
if not archivo.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError("Coloque Online Retail.xlsx junto al notebook.") from exc
    archivos = files.upload()
    candidatos = [nombre for nombre in archivos if nombre.lower().endswith(".xlsx")]
    if len(candidatos) != 1:
        raise ValueError("Cargue un solo Excel: el dataset Online Retail original.")
    archivo = Path(candidatos[0])

df = pd.read_excel(archivo)
df.head()


In [ ]:
print("Filas y columnas:", df.shape)

print("\nColumnas del dataset:")
print(df.columns.tolist())

In [ ]:
# Crear una copia para trabajar sin modificar los datos originales
df_clean = df.copy()

# Eliminar cancelaciones: facturas que comienzan con "C"
df_clean = df_clean[
    ~df_clean["InvoiceNo"].astype(str).str.startswith("C")
]

# Conservar solamente cantidades y precios positivos
df_clean = df_clean[
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0)
]

# Eliminar duplicados exactos
df_clean = df_clean.drop_duplicates()

# Crear el importe de cada línea
df_clean["ImporteLinea"] = (
    df_clean["Quantity"] * df_clean["UnitPrice"]
)

print("Registros originales:", len(df))
print("Registros después de la limpieza:", len(df_clean))

df_clean.head()


In [ ]:
print("Valores faltantes por columna:")
print(df_clean.isnull().sum())

In [ ]:
# Dataset para análisis de clientes
df_clientes = df_clean.dropna(subset=["CustomerID"]).copy()

# Convertir CustomerID a texto
df_clientes["CustomerID"] = (
    df_clientes["CustomerID"]
    .astype(int)
    .astype(str)
)

print("Registros de ventas limpias:", len(df_clean))
print("Registros con cliente identificado:", len(df_clientes))
print("Clientes únicos:", df_clientes["CustomerID"].nunique())

In [ ]:
# Guardar el dataset depurado
df_clientes.to_csv(
    "online_retail_depurado.csv",
    index=False
)

print("Archivo creado correctamente: online_retail_depurado.csv")

In [ ]:
df_clientes[["Quantity", "UnitPrice", "ImporteLinea"]].describe()

### Interpretación del análisis descriptivo

**¿Qué muestra?**  
Las transacciones presentan una distribución desigual. La mediana de la cantidad comprada es de 6 unidades y la mediana del importe por línea es 12.45, mientras que existen valores máximos muy elevados, como 80,995 unidades y un importe de 168,469.60. La diferencia entre la media, la mediana y los máximos evidencia la presencia de valores extremos.

**¿Qué decisión cambiaría?**  
Antes de entrenar los modelos, estos valores deben analizarse y tratarse con precaución para evitar que transacciones excepcionalmente grandes dominen el comportamiento del modelo. En la segmentación de clientes será especialmente importante controlar la escala de las variables.

**¿Qué limitación tiene?**  
Las estadísticas descriptivas no permiten determinar por sí solas si los valores extremos son errores o transacciones reales. Eliminarlos automáticamente podría provocar pérdida de información válida.


In [ ]:
import matplotlib.pyplot as plt

# Crear columna de mes
df_clientes["Mes"] = df_clientes["InvoiceDate"].dt.to_period("M").astype(str)

# Calcular ingresos mensuales
ingresos_mes = df_clientes.groupby("Mes")["ImporteLinea"].sum()

# Crear gráfico
plt.figure(figsize=(12, 5))
ingresos_mes.plot(kind="line", marker="o")

plt.title("Ingresos por mes")
plt.xlabel("Mes")
plt.ylabel("Ingresos")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


### Interpretación de los ingresos mensuales

**¿Qué muestra?**  
Los ingresos presentan variaciones a lo largo del período analizado. Se observa un crecimiento especialmente marcado durante los últimos meses de 2011, alcanzando el nivel más alto en noviembre. Esto sugiere un comportamiento estacional de las ventas y una mayor actividad comercial hacia el cierre del año.

**¿Qué decisión cambiaría?**  
La empresa podría utilizar este patrón para anticipar los meses de mayor demanda, aumentar inventario, preparar capacidad operativa y concentrar campañas comerciales antes del período de mayor actividad.

**¿Qué limitación tiene?**  
El período analizado cubre aproximadamente un año, por lo que no permite confirmar que el mismo patrón se repita anualmente. Además, diciembre de 2011 está incompleto porque el dataset finaliza el 9 de diciembre, por lo que su caída no debe interpretarse directamente como una reducción real de la demanda.

In [ ]:
compras = (
    df_clientes
    .groupby(["CustomerID", "InvoiceNo"], as_index=False)
    .agg(
        FechaCompra=("InvoiceDate", "max"),
        ImporteCompra=("ImporteLinea", "sum"),
        UnidadesCompra=("Quantity", "sum")
    )
)

compras = compras.sort_values(
    ["CustomerID", "FechaCompra"]
)

print("Número de compras/facturas:", len(compras))
print("Clientes únicos:", compras["CustomerID"].nunique())

compras.head(10)

In [ ]:
# Fecha de la siguiente compra del mismo cliente
compras["SiguienteCompra"] = (
    compras.groupby("CustomerID")["FechaCompra"].shift(-1)
)

# Días hasta la siguiente compra
compras["DiasHastaSiguienteCompra"] = (
    compras["SiguienteCompra"] - compras["FechaCompra"]
).dt.days

compras[
    ["CustomerID", "InvoiceNo", "FechaCompra",
     "SiguienteCompra", "DiasHastaSiguienteCompra"]
].head(15)

In [ ]:
# Fecha máxima disponible en el dataset
fecha_maxima = compras["FechaCompra"].max()

# Solo compras con al menos 90 días de seguimiento posible
modelo_supervisado = compras[
    compras["FechaCompra"] <= fecha_maxima - pd.Timedelta(days=90)
].copy()

# Variable objetivo:
# 1 = realizó otra compra dentro de 90 días
# 0 = no realizó otra compra dentro de 90 días
modelo_supervisado["Recompra90"] = (
    modelo_supervisado["DiasHastaSiguienteCompra"]
    .le(90)
    .fillna(False)
    .astype(int)
)

print("Observaciones disponibles:", len(modelo_supervisado))
print("\nDistribución de Recompra90:")
print(modelo_supervisado["Recompra90"].value_counts())

print("\nPorcentajes:")
print(
    modelo_supervisado["Recompra90"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# Variables históricas disponibles antes de cada compra

modelo_supervisado["ComprasPrevias"] = (
    modelo_supervisado.groupby("CustomerID").cumcount()
)

modelo_supervisado["GastoAcumuladoPrevio"] = (
    modelo_supervisado.groupby("CustomerID")["ImporteCompra"]
    .cumsum() - modelo_supervisado["ImporteCompra"]
)

modelo_supervisado["UnidadesAcumuladasPrevias"] = (
    modelo_supervisado.groupby("CustomerID")["UnidadesCompra"]
    .cumsum() - modelo_supervisado["UnidadesCompra"]
)

# Variables que usaremos para predecir
variables = [
    "ImporteCompra",
    "UnidadesCompra",
    "ComprasPrevias",
    "GastoAcumuladoPrevio",
    "UnidadesAcumuladasPrevias"
]

modelo_supervisado[
    variables + ["Recompra90"]
].head(10)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Variables predictoras y variable objetivo
X = modelo_supervisado[variables]
y = modelo_supervisado["Recompra90"]

# Separar entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Línea base: siempre predice la clase más frecuente
baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

pred_baseline = baseline.predict(X_test)

accuracy_baseline = accuracy_score(
    y_test,
    pred_baseline
)

print("Observaciones de entrenamiento:", len(X_train))
print("Observaciones de prueba:", len(X_test))
print("Accuracy línea base:", round(accuracy_baseline, 4))
print("Accuracy línea base (%):", round(accuracy_baseline * 100, 2))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Crear el modelo
arbol = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=20,
    random_state=42
)

# Entrenar el modelo
arbol.fit(X_train, y_train)

# Realizar predicciones
pred_arbol = arbol.predict(X_test)

# Calcular accuracy
accuracy_arbol = accuracy_score(
    y_test,
    pred_arbol
)

print("Accuracy línea base (%):",
      round(accuracy_baseline * 100, 2))

print("Accuracy árbol de decisión (%):",
      round(accuracy_arbol * 100, 2))

print("Mejora sobre línea base:",
      round((accuracy_arbol - accuracy_baseline) * 100, 2),
      "puntos porcentuales")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_arbol,
    display_labels=["No recompra", "Recompra"],
    cmap="Blues"
)

plt.title("Matriz de confusión – Árbol de decisión")
plt.tight_layout()
plt.show()

### Interpretación de la matriz de confusión

**¿Qué muestra?**  
El árbol de decisión clasificó correctamente 1,590 casos de recompra y 187 casos de no recompra. Sin embargo, clasificó incorrectamente 448 casos como recompra cuando realmente no ocurrió y 157 casos como no recompra cuando sí ocurrió. El modelo alcanzó una exactitud de 74.60%, superando la línea base de 73.34% en 1.26 puntos porcentuales.

**¿Qué decisión cambiaría?**  
El modelo podría utilizarse de manera exploratoria para identificar clientes con mayor probabilidad de volver a comprar y orientar acciones de fidelización. Sin embargo, su capacidad para distinguir clientes que no recomprarán todavía es limitada.

**¿Qué limitación tiene?**  
La mejora frente a la línea base es pequeña y el modelo presenta una cantidad considerable de falsos positivos. Por esta razón, los resultados todavía no justifican su implementación automática en producción y requieren mayor validación y optimización.

In [ ]:

# Fecha de referencia: un día después de la última compra
fecha_referencia = df_clientes["InvoiceDate"].max() + pd.Timedelta(days=1)

# Crear tabla RFM por cliente
rfm = (
    df_clientes
    .groupby("CustomerID")
    .agg(
        Recencia=("InvoiceDate",
                  lambda x: (fecha_referencia - x.max()).days),
        Frecuencia=("InvoiceNo", "nunique"),
        Monetario=("ImporteLinea", "sum")
    )
    .reset_index()
)

print("Clientes incluidos:", len(rfm))

rfm.head(10)

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Variables RFM
X_rfm = rfm[["Recencia", "Frecuencia", "Monetario"]].copy()

# Transformación logarítmica para reducir el efecto de valores extremos
X_rfm_log = np.log1p(X_rfm)

# Estandarizar las variables
scaler = StandardScaler()
X_rfm_scaled = scaler.fit_transform(X_rfm_log)

print("Dimensiones de los datos preparados:", X_rfm_scaled.shape)
print("Datos preparados correctamente para K-Means.")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

resultados_silueta = {}

for k in range(2, 7):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    etiquetas = kmeans.fit_predict(X_rfm_scaled)

    score = silhouette_score(
        X_rfm_scaled,
        etiquetas
    )

    resultados_silueta[k] = score

for k, score in resultados_silueta.items():
    print(f"k = {k} | Silhouette Score = {score:.4f}")

In [ ]:
# Modelo K-Means final con 2 segmentos
kmeans_final = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

rfm["Cluster"] = kmeans_final.fit_predict(X_rfm_scaled)

# Perfil de cada segmento
perfil_clusters = (
    rfm.groupby("Cluster")
    .agg(
        Clientes=("CustomerID", "count"),
        Recencia_Media=("Recencia", "mean"),
        Frecuencia_Media=("Frecuencia", "mean"),
        Monetario_Medio=("Monetario", "mean")
    )
    .round(2)
)

perfil_clusters

### Interpretación de la segmentación de clientes

**¿Qué muestra?**  
El modelo K-Means identificó dos segmentos claramente diferenciados. El Cluster 0 contiene 2,672 clientes con una recencia promedio de 134.09 días, una frecuencia de 1.67 compras y un valor monetario promedio de 495.59, representando clientes de baja actividad. El Cluster 1 contiene 1,666 clientes con una recencia promedio de 25.89 días, una frecuencia de 8.44 compras y un gasto promedio de 4,539.60, por lo que representa clientes activos y de alto valor.

La selección de dos segmentos se sustentó en el Silhouette Score. K=2 obtuvo el valor más alto (0.4328), superior a las alternativas evaluadas entre 3 y 6 clusters.

**¿Qué decisión cambiaría?**  
La empresa podría aplicar estrategias diferenciadas. Para el Cluster 1 se recomienda priorizar acciones de fidelización, beneficios exclusivos y recomendaciones personalizadas. Para el Cluster 0 podrían utilizarse campañas de reactivación, promociones y comunicaciones dirigidas a estimular nuevas compras.

**¿Qué limitación tiene?**  
La segmentación se basa exclusivamente en recencia, frecuencia y valor monetario. No incorpora características demográficas, preferencias de producto u otras variables externas. Además, los segmentos reflejan el comportamiento observado durante el período disponible y podrían cambiar con nuevos datos.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))

for cluster in sorted(rfm["Cluster"].unique()):
    datos = rfm[rfm["Cluster"] == cluster]

    plt.scatter(
        datos["Frecuencia"],
        datos["Monetario"],
        alpha=0.5,
        label=f"Cluster {cluster}"
    )

plt.xscale("log")
plt.yscale("log")

plt.title("Segmentación de clientes mediante K-Means")
plt.xlabel("Frecuencia de compra")
plt.ylabel("Valor monetario")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretación visual de los segmentos

**¿Qué muestra?**  
El gráfico confirma la diferenciación entre los dos segmentos identificados mediante K-Means. El Cluster 0 se concentra principalmente en clientes con baja frecuencia de compra y menor valor monetario, mientras que el Cluster 1 agrupa mayoritariamente clientes con compras más frecuentes y niveles de gasto superiores.

**¿Qué decisión cambiaría?**  
Esta separación permite diseñar estrategias comerciales diferenciadas. Los clientes del Cluster 1 pueden recibir acciones orientadas a fidelización y mantenimiento de su valor, mientras que el Cluster 0 puede ser objeto de campañas de reactivación y estímulos para aumentar su frecuencia de compra.

**¿Qué limitación tiene?**  
Aunque existe una separación general entre los segmentos, también se observa cierto solapamiento entre algunos clientes. Por ello, la pertenencia a un cluster no debe considerarse una clasificación absoluta del comportamiento futuro del cliente.

## Juicio profesional sobre la implementación en producción

Los resultados obtenidos demuestran que es técnicamente viable aplicar técnicas de aprendizaje automático sobre los datos de Online Retail para apoyar decisiones relacionadas con la recompra y la segmentación de clientes. Sin embargo, en el estado actual no se recomienda implementar el modelo supervisado directamente en un entorno de producción.

El árbol de decisión alcanzó una exactitud de 74.60%, frente al 73.34% obtenido por la línea base, lo que representa una mejora de solo 1.26 puntos porcentuales. La matriz de confusión mostró 1,590 recompras correctamente identificadas y 187 no recompras correctamente clasificadas, pero también 448 falsos positivos y 157 falsos negativos. Estos resultados indican que el modelo tiene capacidad predictiva, pero la mejora respecto a una estrategia básica todavía es limitada para justificar decisiones automatizadas.

Por otra parte, la segmentación mediante K-Means presenta mayor utilidad inmediata como herramienta de apoyo empresarial. El mejor resultado se obtuvo con dos clusters y un Silhouette Score de 0.4328. El Cluster 0 agrupó 2,672 clientes de menor actividad, mientras que el Cluster 1 identificó 1,666 clientes más frecuentes, recientes y de mayor valor monetario. Esta información puede utilizarse para diseñar estrategias diferenciadas de reactivación y fidelización.

Antes de considerar una implementación en producción se recomienda incorporar más períodos históricos, realizar validaciones temporales adicionales, evaluar nuevas variables y algoritmos, optimizar el modelo supervisado y establecer mecanismos de monitoreo y reentrenamiento. En consecuencia, la solución actual se recomienda como prototipo analítico y herramienta de apoyo a decisiones, pero no todavía como sistema automatizado de producción.

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 10))

plot_tree(
    arbol,
    feature_names=variables,
    class_names=["No recompra", "Recompra"],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Árbol de decisión para predicción de recompra")
plt.tight_layout()
plt.show()

### Interpretación del árbol de decisión

**¿Qué muestra?**  
El árbol de decisión utiliza el comportamiento de compra del cliente para identificar patrones asociados con la recompra dentro de los siguientes 90 días. Entre las variables utilizadas se encuentran el gasto acumulado previo, el número de compras anteriores, el importe de la compra y las unidades adquiridas. El gasto acumulado previo aparece desde la primera división del árbol, indicando su relevancia para separar los perfiles de clientes.

**¿Qué decisión cambiaría?**  
El modelo permite identificar características asociadas con una mayor probabilidad de recompra y puede servir como apoyo para seleccionar clientes en campañas de fidelización y seguimiento comercial.

**¿Qué limitación tiene?**  
Aunque el árbol facilita la interpretación de las reglas de decisión, su exactitud fue de 74.60%, apenas 1.26 puntos porcentuales superior a la línea base. Por tanto, sus reglas deben considerarse exploratorias y no utilizarse todavía como único criterio para automatizar decisiones comerciales.